# Pretrained Model Linear Pruning Experiments (IEEE-Style Figures)

This notebook loads all available pretrained models, runs linear/magnitude pruning on each model in separate experiment cells, and plots NMSE versus pruning percentage in a publication-ready format.

In [1]:
# Imports and plotting defaults
import os
import numpy as np
import matplotlib.pyplot as plt

from sparseDPD import DataManager
from sparseDPD import LinearExperiment
from sparseDPD import ARVTDNN_NeuralNetwork
from sparseDPD import PNTDNN_NeuralNetwork
from sparseDPD import PGJANET_NeuralNetwork

# IEEE-style plotting defaults
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
    'mathtext.fontset': 'dejavuserif',
    'font.size': 9,
    'axes.labelsize': 9,
    'axes.titlesize': 9,
    'legend.fontsize': 8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'lines.linewidth': 1.6,
    'lines.markersize': 4,
    'figure.dpi': 160,
    'savefig.dpi': 300
})

print('Imports loaded.')

Imports loaded.


In [2]:
# Load dataset
data_manager = DataManager(
    filepath='UCD_datasets/PA_IO.mat',
    num_training_points=19000,
    num_validaiton_points=1000,
    num_test_points=2000
)

training_dataset = data_manager.training_dataset
validation_dataset = data_manager.validation_dataset
test_dataset = data_manager.test_dataset

print('Dataset ready.')

Dataset ready.


In [3]:
# Load all pretrained models
models = {}

models['ARVTDNN_OneLayer'] = ARVTDNN_NeuralNetwork(
    num_memory_levels=15,
    forward_model=True,
    model_type='OneLayerNetwork',
    nn_file_path='pre_trained_models/arvtdnn_one_layer.pt'
)

# Keep PGJANET configuration aligned with the baseline training notebook.
models['PGJANET'] = PGJANET_NeuralNetwork(
    num_memory_levels=50,
    model_type='PGJANETNetwork',
    forward_model=True,
    seq_stride=1,
    batch_size=32,
    hidden_size=32,
    nn_file_path='pre_trained_models/pgjanet_checkpoint.pt'
)

models['PNTDNN_Checkpoint'] = PNTDNN_NeuralNetwork(
    num_memory_levels=15,
    forward_model=True,
    nn_file_path='pre_trained_models/pntdnn_checkpoint.pt'
)

models['PNTDNN_OneLayer'] = PNTDNN_NeuralNetwork(
    num_memory_levels=15,
    forward_model=True,
    model_type='OneLayerNetwork',
    nn_file_path='pre_trained_models/pntdnn_one_layer.pt'
)

models['PNTDNN_ThreeLayer'] = PNTDNN_NeuralNetwork(
    num_memory_levels=15,
    forward_model=True,
    model_type='ThreeLayerNetwork',
    nn_file_path='pre_trained_models/pntdnn_three_layer.pt'
)

print('Loaded models:', ', '.join(models.keys()))

Using cpu device
Using cpu device
Using cpu device
Using cpu device
Using cpu device
Loaded models: ARVTDNN_OneLayer, PGJANET, PNTDNN_Checkpoint, PNTDNN_OneLayer, PNTDNN_ThreeLayer


In [4]:
# Common pruning settings and result storage
num_prune_iterations = 12
prune_amount = 0.2

# PGJANET converges faster, so use fewer retraining epochs than other models.
default_retrain_epochs = 100
model_retrain_epochs = {
    'PGJANET': 35
}

linear_pruning_results = {}

def run_linear_pruning_experiment(model_name):
    retrain_epochs = model_retrain_epochs.get(model_name, default_retrain_epochs)

    exp = LinearExperiment(
        nn_model=models[model_name],
        num_prune_iterations=num_prune_iterations,
        prune_amount=prune_amount,
        retrain_epochs=retrain_epochs,
        training_dataset=training_dataset,
        valid_dataset=validation_dataset,
        test_dataset=test_dataset
    )

    baseline_nmse = exp.original_nn_model.calculate_forward_nmse(exp.test_dataset)
    baseline_params = exp.original_nn_model.get_num_params()

    prune_pcts, nmse_results, valid_losses, best_epochs, all_valid_losses = exp.prune()

    linear_pruning_results[model_name] = {
        'baseline_nmse': baseline_nmse,
        'baseline_params': baseline_params,
        'retrain_epochs': retrain_epochs,
        'prune_pcts': [0.0] + prune_pcts,
        'nmse': [baseline_nmse] + nmse_results,
        'valid_losses': all_valid_losses,
        'best_epochs': best_epochs
    }

    print(f"{model_name}: retrain_epochs={retrain_epochs}")
    print(f"{model_name}: baseline NMSE={baseline_nmse:.2f} dB, params={baseline_params:,}")
    print(f"{model_name}: final pruning={linear_pruning_results[model_name]['prune_pcts'][-1]:.2f}%")

## Linear Pruning Experiments (One Cell per Model)

In [ ]:
# Experiment 1: ARVTDNN_OneLayer
run_linear_pruning_experiment('ARVTDNN_OneLayer')


Pruning Iteration 1/12
Pruning 20.0% of remaining weights...
Current pruning: 20.00% of weights are zero
Retraining for 100 epochs...


In [ ]:
# Experiment 2: PGJANET
run_linear_pruning_experiment('PGJANET')

In [ ]:
# Experiment 3: PNTDNN_Checkpoint
run_linear_pruning_experiment('PNTDNN_Checkpoint')

In [ ]:
# Experiment 4: PNTDNN_OneLayer
run_linear_pruning_experiment('PNTDNN_OneLayer')

In [ ]:
# Experiment 5: PNTDNN_ThreeLayer
run_linear_pruning_experiment('PNTDNN_ThreeLayer')

In [ ]:
# IEEE-style summary figure: NMSE vs Pruning Percentage
fig, ax = plt.subplots(figsize=(7.1, 3.2))  # Two-column friendly

style_map = {
    'ARVTDNN_OneLayer': {'marker': 'o', 'linestyle': '-', 'color': '#1f77b4'},
    'PGJANET': {'marker': 's', 'linestyle': '-', 'color': '#ff7f0e'},
    'PNTDNN_Checkpoint': {'marker': 'D', 'linestyle': '--', 'color': '#2ca02c'},
    'PNTDNN_OneLayer': {'marker': '^', 'linestyle': '--', 'color': '#d62728'},
    'PNTDNN_ThreeLayer': {'marker': 'P', 'linestyle': ':', 'color': '#8c564b'}
}

for model_name, result in linear_pruning_results.items():
    prune_pct = result['prune_pcts']
    nmse = result['nmse']
    base_params = result['baseline_params']

    style = style_map.get(model_name, {'marker': 'o', 'linestyle': '-', 'color': 'black'})
    label = f"{model_name} ({base_params:,} params)"

    ax.plot(
        prune_pct,
        nmse,
        marker=style['marker'],
        linestyle=style['linestyle'],
        color=style['color'],
        label=label
    )

ax.set_xlabel('Pruning Percentage (%)')
ax.set_ylabel('NMSE (dB)')
ax.set_title('Pretrained Models: Linear Pruning Performance')
ax.grid(True, which='major', linestyle='--', linewidth=0.6, alpha=0.6)
ax.set_xlim(left=-1)
ax.legend(loc='best', ncol=2, frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
# Optional: Save high-resolution figure for paper usage
fig, ax = plt.subplots(figsize=(7.1, 3.2))

for model_name, result in linear_pruning_results.items():
    style = style_map.get(model_name, {'marker': 'o', 'linestyle': '-', 'color': 'black'})
    ax.plot(
        result['prune_pcts'],
        result['nmse'],
        marker=style['marker'],
        linestyle=style['linestyle'],
        color=style['color'],
        label=model_name
    )

ax.set_xlabel('Pruning Percentage (%)')
ax.set_ylabel('NMSE (dB)')
ax.set_title('NMSE vs Pruning Percentage (Linear Pruning)')
ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
ax.legend(loc='best', ncol=2, frameon=True)

plt.tight_layout()
output_path = 'pretrained_linear_pruning_ieee.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved figure: {output_path}')